In [14]:
homedir = '/mnt/mirabelle/az6922_homedir/DRing/src/emp/datacentre/'
import random
import numpy as np

# from makec2s.ipynb
def genflowbytes():
    np.random.seed(0)
    
    mean_bytes = 100.0 * 1024
    shape = 1.05
    scale = mean_bytes * (shape - 1)/shape

    x = np.random.exponential(scale=1.0/shape)
    flowbytes = int(scale * np.exp(x))
    return flowbytes

def adjustbytesbymtu(flowbytes):
  mss = 1500
  return mss * ((flowbytes+mss-1)//mss)

large_flow_threshold = 10 * 1024 * 1024

In [16]:
stime = 144 # ms
nlinks = 2132 # 1066*2, uni-directional
nhosts = 2988
bw = 1342176000 # B per second
# load_list = range(1,11) # [10,30,50,70]
# seed_list = [1,2,3,4,5]
topologytype = 2
nswitches = 80
os = 1
k = 64
nintervals = 8
topologyfile = 'evaltopologyfiles/dring_80_64.edgelist'
serverfile = 'evalserverfiles/dring_2988_80_64.sv'
npfile = 'evalnetpathfiles/netpath_dring_80_64_su2.np'
failpct_list = range(2,11,2)
load = 5
seed = 1
failseed_list = range(10)

In [6]:
# generate connection_matrices file (1)
unv1bytes = 0
unv1file = f'{homedir}rawtrafficfiles/unv1'
maxinterval = 0
with open(unv1file, 'r') as f:
    lines = f.readlines()
    for line in lines:
        tokens = line.split(',')
        # 0,32,31,10500
        # interval,fromserver,toserver,bytes
        unv1bytes += int(tokens[3])
        maxinterval = max(maxinterval, int(tokens[0]))
print(f'unv1bytes {unv1bytes}, maxinterval {maxinterval}, fullload {bw * stime * nlinks / 1000}, ratio {(bw * stime * nlinks / 1000) / unv1bytes}')

unv1bytes 162036861000, maxinterval 7, fullload 412058769408.0, ratio 2.5429940253409375


In [7]:
# generate connection_matrices file (2)
random.seed(0)
totalbytes = bw * stime / 1000 * nlinks * load / 100  # B

serverfile = f"{homedir}evalserverfiles/dring_2988_80_64.sv"
servertoswdict = dict()
with open(serverfile, 'r') as f:
    lines = f.readlines()
    for line in lines:
        tokens = line.split(',')
        serverid = int(tokens[0])
        switchid = int(tokens[1])
        servertoswdict[serverid] = switchid

for failpct in failpct_list:
    numfailtor = int(nswitches * failpct / 100)
    for failseed in failseed_list:
        torfailurefile = f"{homedir}/experiments/nsdi26fall/eval_failure_tor/torfailurefiles/leafspine_{numfailtor}_{failseed}.lf"
        torlist = list()
        with open(torfailurefile, 'r') as f:
            lines = f.readlines()
            for line in lines:
                torlist.append(int(line))

        cmfile = f'cmfiles/dring_load{load}_{numfailtor}_{failseed}.cm'
        mult = totalbytes / unv1bytes
        actualbytes = 0
        with open(cmfile, 'w') as fw:
            with open(unv1file, 'r') as fr:
                lines = fr.readlines()
                iline = 0
                while actualbytes < totalbytes:
                    line = lines[iline]
                    tokens = line.split(',')
                    interval = int(tokens[0])
                    fromserver = int(tokens[1])
                    toserver = int(tokens[2])
                    multbytes = int(tokens[3])

                    if fromserver >= nhosts or toserver >= nhosts:
                        iline += 1
                        if iline >= len(lines):
                            iline = 0
                            if mult-1>0:
                                mult = mult-1
                        continue

                    if mult >= 1 or (random.random() < mult):
                        multbytes = adjustbytesbymtu(multbytes)
        
                        # generate random start time
                        start_time_ms = random.uniform(0, stime//(maxinterval+1)) + interval * (stime//(maxinterval+1))

                        actualbytes += int(multbytes)
                        
                        fromsw = servertoswdict[fromserver]
                        tosw = servertoswdict[toserver]
                        if fromsw not in torlist and tosw not in torlist:
                            fw.write(f'{fromserver},{toserver},{int(multbytes)},{start_time_ms:.4f}\n')
                        

                    iline += 1
                    if iline >= len(lines):
                        iline = 0
                        if mult-1>0:
                            mult = mult-1

                        # print(f'actualbytes {actualbytes}, totalbytes {totalbytes}, mult {mult}', end='\r')

        # print(f'load {load}%, totalbytes {totalbytes}, unv1bytes {unv1bytes}, mult {mult}, actualbytes {actualbytes}')


In [ ]:
# # generate pathweight file (1)
# interval_stime = stime / nintervals
# with open('dringsu2_generate_pwfiles.conf', 'w') as f:
#     for load in load_list:
#         cmfile = f'cmfiles/dring_load{load}.cm'
#         for interval in range(nintervals):
#             flowstart = interval_stime * interval
#             flowend = interval_stime * (interval + 1)
#             varfile = f'{homedir}rawpathweightfiles/pathtraffic_dring_{nhosts}_{nswitches}_{k}_su2_unv1_load{load}_interval{interval}.var'
#             qvarfile = f'{homedir}rawpathweightfiles/pathweight_dring_{nhosts}_{nswitches}_{k}_su2_unv1_load{load}_interval{interval}.var'
#             f.write(f"python3 {homedir}generate_pathweightfiles.py --graphfile {homedir}{topologyfile} --serverfile {homedir}{serverfile} --numsw {nswitches} --numserver {nhosts} --netpathfile {homedir}{npfile} --flowfile {cmfile} --flowstart {flowstart} --flowend {flowend} --numfaillink 0 --linkfailurefile none --varfile {varfile} --qvarfile {qvarfile}\n")

~~(current dir: ~/DRing/src/emp/datacentre/experiments/nsdi26fall/eval_main/unv1/)
python3 ../../../../pararun.py --conf dringsu2_generate_pwfiles.conf --worker 72~~

In [10]:
# generate pathweight file (2)
intervaldict = {0:0,1:0,2:1,3:2,4:3,5:4,6:5,7:6} # to:from
with open('dringsu2_before_generate_pwfiles.conf', 'w') as fwconf:
    for failpct in failpct_list:
        numfailtor = int(nswitches * failpct / 100)
        for failseed in failseed_list:
            torfailurefile = f"{homedir}/experiments/nsdi26fall/eval_failure_tor/torfailurefiles/leafspine_{numfailtor}_{failseed}.lf"
            for interval in range(nintervals):
                fromfile = f'{homedir}rawpathweightfiles/pathweight_dring_{nhosts}_{nswitches}_{k}_su2_unv1_load{load}_interval{intervaldict[interval]}.var'
                tofile = f'{homedir}experiments/nsdi26fall/eval_failure_tor/unv1/pwfiles/pathweight_dring_su2_unv1_before_load{load}_{numfailtor}_{failseed}_interval{interval}.pw'
                fwconf.write(f'python3 {homedir}generate_dring_before_torfailurepathweightfiles.py --numsw {nswitches} --torfailurefile {torfailurefile} --netpathfile {npfile} --beforepathweightfile {fromfile} --afterpathweightfile {tofile}\n')

(current dir: ~/DRing/src/emp/datacentre/)
python3 pararun.py --conf experiments/nsdi26fall/eval_failure_tor/unv1/dringsu2_before_generate_pwfiles.conf --worker 72

In [ ]:
# # generate linkfailurefiles
# with open(f"{homedir}experiments/nsdi26fall/eval_failure_link/dring_lffiles.conf", 'w') as f:
#     for failpct in failpct_list:
#         numfaillinks = int(nlinks * failpct / 100)
#         for failseed in failseed_list:
#             linkfailurefile = f"{homedir}/experiments/nsdi26fall/eval_failure_link/linkfailurefiles/dring_{numfaillinks}_{failseed}.lf"
#             f.write(f"python3 {homedir}generate_dring_linkfailurefiles.py --numfaillinks {numfaillinks} --rseed {failseed} --linkfailurefile {linkfailurefile}\n")

~~(current dir: ~/DRing/src/emp/datacenter/experiments/nsdi26fall/eval_failure_link/)
python3 ../../../pararun.py --conf dring_lffiles.conf --worker 20~~

In [9]:
# generate conf file
conffile = f'{homedir}experiments/nsdi26fall/eval_failure_tor/unv1/run_dringsu2_before.conf'
with open(conffile, 'w') as f:
    for failpct in failpct_list:
        numfailtor = int(nswitches * failpct / 100)
        for failseed in failseed_list:
            pwfileprefix = f'experiments/nsdi26fall/eval_failure_tor/unv1/pwfiles/pathweight_dring_su2_unv1_before_load{load}_{numfailtor}_{failseed}_interval'
            cmfile = f'experiments/nsdi26fall/eval_failure_tor/unv1/cmfiles/dring_load{load}_{numfailtor}_{failseed}.cm'
            outfile = f'experiments/nsdi26fall/eval_failure_tor/unv1/outfiles/dringsu2_before_nfailtor{numfailtor}_fseed{failseed}.out'
            f.write(f"./eval -stime {stime} -seed {seed} -cmfile {cmfile} -topologytype {topologytype} -numswitches {nswitches} -numhosts {nhosts} -os {os} -ls_k {k} -npfile {npfile} -pwfileprefix {pwfileprefix} -numintervals {nintervals} -serverfile {serverfile} -topologyfile {topologyfile} > {outfile}\n")
            

(current dir: ~/DRing/src/emp/datacentre/)
python3 pararun.py --conf experiments/nsdi26fall/eval_failure_tor/unv1/run_dringsu2_before.conf --worker 50

================================================

In [18]:
# generate pathweight file (1)
interval_stime = stime / nintervals
with open('dringsu2_after_generate_pwfiles.conf', 'w') as f:
    for failpct in failpct_list:
        numfailtor = int(nswitches * failpct / 100)
        for failseed in failseed_list:
            torfailurefile = f"{homedir}/experiments/nsdi26fall/eval_failure_tor/torfailurefiles/leafspine_{numfailtor}_{failseed}.lf"
            cmfile = f'{homedir}/experiments/nsdi26fall/eval_failure_tor/unv1/cmfiles/dring_load{load}_{numfailtor}_{failseed}.cm'
            for interval in range(nintervals):
                flowstart = interval_stime * interval
                flowend = interval_stime * (interval + 1)
                varfile = f'{homedir}rawpathweightfiles/pathtraffic_dring_{nhosts}_{nswitches}_{k}_su2_unv1_nfailtor{numfailtor}_fseed{failseed}_load{load}_{numfailtor}_{failseed}_interval{interval}.var'
                qvarfile = f'{homedir}rawpathweightfiles/pathweight_dring_{nhosts}_{nswitches}_{k}_su2_unv1_nfailtor{numfailtor}_fseed{failseed}_load{load}_{numfailtor}_{failseed}_interval{interval}.var'
                f.write(f"python3 {homedir}generate_dring_after_torfailurepathweightfiles.py --graphfile {homedir}{topologyfile} --serverfile {homedir}{serverfile} --numsw {nswitches} --numserver {nhosts} --netpathfile {homedir}{npfile} --flowfile {cmfile} --flowstart {flowstart} --flowend {flowend} --torfailurefile {torfailurefile} --varfile {varfile} --qvarfile {qvarfile}\n")

(current dir: ~/DRing/src/emp/datacentre/experiments/nsdi26fall/eval_failure_tor/unv1/)
python3 ../../../../pararun.py --conf dringsu2_after_generate_pwfiles.conf --worker 80

In [20]:
# generate pathweight file (2)
intervaldict = {0:0,1:0,2:1,3:2,4:3,5:4,6:5,7:6} # to:from
with open('dringsu2_after_copy_pwfiles.conf', 'w') as f:
    for failpct in failpct_list:
        numfailtor = int(nswitches * failpct / 100)
        for failseed in failseed_list:
            for interval in range(nintervals):
                fromfile = f'{homedir}rawpathweightfiles/pathweight_dring_{nhosts}_{nswitches}_{k}_su2_unv1_nfailtor{numfailtor}_fseed{failseed}_load{load}_{numfailtor}_{failseed}_interval{intervaldict[interval]}.var'
                tofile = f'{homedir}experiments/nsdi26fall/eval_failure_tor/unv1/pwfiles/pathweight_dring_su2_unv1_nfailtor{numfailtor}_fseed{failseed}_load{load}_interval{interval}.pw'
                f.write(f'cp {fromfile} {tofile}\n')

actually run the copy commands in datacentre/

In [21]:
# generate conf file
conffile = f'{homedir}experiments/nsdi26fall/eval_failure_tor/unv1/run_dringsu2_after.conf'
with open(conffile, 'w') as f:
    for failpct in failpct_list:
        numfailtor = int(nswitches * failpct / 100)
        for failseed in failseed_list:
            pwfileprefix = f'experiments/nsdi26fall/eval_failure_tor/unv1/pwfiles/pathweight_dring_su2_unv1_nfailtor{numfailtor}_fseed{failseed}_load{load}_interval'
            cmfile = f'experiments/nsdi26fall/eval_failure_tor/unv1/cmfiles/dring_load{load}_{numfailtor}_{failseed}.cm'
            outfile = f'experiments/nsdi26fall/eval_failure_tor/unv1/outfiles/dringsu2_after_nfailtor{numfailtor}_fseed{failseed}.out'
            f.write(f"./eval -stime {stime} -seed {seed} -cmfile {cmfile} -topologytype {topologytype} -numswitches {nswitches} -numhosts {nhosts} -os {os} -ls_k {k} -npfile {npfile} -pwfileprefix {pwfileprefix} -numintervals {nintervals} -serverfile {serverfile} -topologyfile {topologyfile} > {outfile}\n")
            

(current dir: ~/DRing/src/emp/datacentre/)
python3 pararun.py --conf experiments/nsdi26fall/eval_failure_tor/unv1/run_dringsu2_after.conf --worker 50